In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [52]:
from google.colab import files
uploaded = files.upload()
df = pd.read_csv('human_decision_fatigue_dataset.csv')
df.head()

Saving human_decision_fatigue_dataset.csv to human_decision_fatigue_dataset (4).csv


,Hours_Awake,Decisions_Made,Task_Switches,Avg_Decision_Time_sec,Sleep_Hours_Last_Night,Time_of_Day,Caffeine_Intake_Cups,Stress_Level_1_10,Error_Rate,Cognitive_Load_Score,Decision_Fatigue_Score,Fatigue_Level,System_Recommendation
0,7,28,7,2.30,5.8,Evening,0,2.4,0.000,2.6,15.6,Low,Continue
1,15,77,22,3.65,4.5,Afternoon,3,1.9,0.143,4.5,97.3,High,Take Break
2,11,57,23,3.67,6.8,Night,2,2.1,0.000,4.1,55.4,Moderate,Slow Down
3,8,39,10,2.39,5.3,Afternoon,1,1.0,0.000,2.3,29.7,Low,Continue
4,7,46,16,3.05,8.2,Night,1,2.8,0.000,3.9,19.1,Low,Continue


In [53]:
# Drop unwanted columns
df = df.drop([
    'System_Recommendation',
    'Error_Rate',
    'Decision_Fatigue_Score'
], axis=1)

# Check remaining columns
df.columns

Index(['Hours_Awake', 'Decisions_Made', 'Task_Switches',
       'Avg_Decision_Time_sec', 'Sleep_Hours_Last_Night', 'Time_of_Day',
       'Caffeine_Intake_Cups', 'Stress_Level_1_10', 'Cognitive_Load_Score',
       'Fatigue_Level'],
      dtype='object')

HANDLE MISSING VALUES

In [54]:
# Identify columns
categorical_cols = df.select_dtypes(include=['object']).columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

# Fill missing values
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

for col in numerical_cols:
    df[col] = df[col].fillna(df[col].mean())

print("Missing values handled")

Missing values handled


In [55]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Check if the one-hot encoded Fatigue_Level columns exist
if 'Fatigue_Level_Low' in df.columns and 'Fatigue_Level_Moderate' in df.columns:
    # Reconstruct the original 'Fatigue_Level' column
    def reconstruct_fatigue_level(row):
        if row['Fatigue_Level_Low']:
            return 'Low'
        elif row['Fatigue_Level_Moderate']:
            return 'Moderate'
        else:
            # Assuming 'High' is the implied third category when both Low and Moderate are False
            return 'High'

    df['Fatigue_Level_Original'] = df.apply(reconstruct_fatigue_level, axis=1)

    # Encode the reconstructed target variable
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(df['Fatigue_Level_Original'])

    # Drop the one-hot encoded columns and the reconstructed original 'Fatigue_Level' from features
    X = df.drop(columns=['Fatigue_Level_Low', 'Fatigue_Level_Moderate', 'Fatigue_Level_Original'], axis=1)

    print("Target and features separated, and 'Fatigue_Level' reconstructed and encoded from one-hot columns.")
else:
    # Fallback if the columns are not found in the expected one-hot encoded format
    print("Error: The column 'Fatigue_Level' or its expected one-hot encoded forms (Fatigue_Level_Low, Fatigue_Level_Moderate) were not found in the DataFrame.")
    print("Current columns in DataFrame:")
    print(df.columns.tolist())
    print("Please ensure 'Fatigue_Level' exists in the DataFrame in an appropriate format for processing.")

Error: The column 'Fatigue_Level' or its expected one-hot encoded forms (Fatigue_Level_Low, Fatigue_Level_Moderate) were not found in the DataFrame.
Current columns in DataFrame:
['Hours_Awake', 'Decisions_Made', 'Task_Switches', 'Avg_Decision_Time_sec', 'Sleep_Hours_Last_Night', 'Time_of_Day', 'Caffeine_Intake_Cups', 'Stress_Level_1_10', 'Cognitive_Load_Score', 'Fatigue_Level']
Please ensure 'Fatigue_Level' exists in the DataFrame in an appropriate format for processing.


In [56]:
# Encode categorical input features
categorical_X = X.select_dtypes(include=['object']).columns
X = pd.get_dummies(X, columns=categorical_X, drop_first=True)

# Encode target labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

print("Encoding done")

Encoding done


In [57]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Data split completed")

Data split completed


In [58]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

rf = RandomForestClassifier(random_state=42)

grid = GridSearchCV(rf, param_grid, cv=3, n_jobs=-1)
grid.fit(X_train, y_train)

model = grid.best_estimator_

print("Best Parameters:", grid.best_params_)

Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}


In [59]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9774

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.99      0.99      1692
           1       0.99      0.99      0.99      2336
           2       0.95      0.93      0.94       972

    accuracy                           0.98      5000
   macro avg       0.97      0.97      0.97      5000
weighted avg       0.98      0.98      0.98      5000



In [60]:
# =========================
# SHOW ENCODING OF FATIGUE LEVEL
# =========================

print("Fatigue Level Encoding:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{label} -> {i}")

# =========================
# TAKE SAMPLE FROM TEST SET
# =========================

sample_input = X_test.iloc[0]   # already from test data
actual_label = y_test[0]

# Convert to DataFrame
import pandas as pd
input_df = pd.DataFrame([sample_input])

# =========================
# PREDICT
# =========================

pred = model.predict(input_df)[0]

# Decode labels
pred_label = label_encoder.inverse_transform([pred])[0]
actual_label_decoded = label_encoder.inverse_transform([actual_label])[0]

# =========================
# RECOMMENDATION
# =========================

if pred_label == "Low":
    recommendation = "Safe to work"
elif pred_label == "Medium":
    recommendation = "Work with caution"
else:
    recommendation = "Rest recommended"

# =========================
# PRINT OUTPUT
# =========================

print("\n--- SAMPLE OUTPUT ---")
print("Actual Fatigue Level:", actual_label_decoded)
print("Predicted Fatigue Level:", pred_label)
print("Recommendation:", recommendation)


Fatigue Level Encoding:
0 -> 0
1 -> 1
2 -> 2

--- SAMPLE OUTPUT ---
Actual Fatigue Level: 0
Predicted Fatigue Level: 0
Recommendation: Rest recommended


Here 0:High fatigue,1:low,2:medium

In [61]:
def optimize_task_switches_practical(input_data):

    import pandas as pd

    best_switch = None
    best_score = float('inf')
    best_pred = None

    fatigue_rank = {
        1: 0,  # Low
        2: 1,  # Medium
        0: 2   # High
    }

    current_switch = input_data.get('Task_Switches', 5)

    # search around current value (key idea)
    search_range = range(max(2, current_switch - 5), current_switch + 6)

    for switches in search_range:

        temp_input = input_data.copy()
        temp_input['Task_Switches'] = switches

        input_df = pd.DataFrame([temp_input])
        input_df = pd.get_dummies(input_df)
        input_df = input_df.reindex(columns=X.columns, fill_value=0)

        pred = model.predict(input_df)[0]
        fatigue_score = fatigue_rank[pred]

        # penalty for extreme deviation
        deviation_penalty = abs(switches - current_switch) * 0.2

        total_score = fatigue_score + deviation_penalty

        if total_score < best_score:
            best_score = total_score
            best_switch = switches
            best_pred = pred

    return best_switch, best_pred

In [62]:
# Mapping
fatigue_map = {
    0: "High",
    1: "Low",
    2: "Medium"
}

# Test on multiple samples
for i in range(5):   # change number of samples here

    sample_input = X_test.iloc[i].to_dict()
    actual_label = y_test[i]

    # Step 1: Predict current fatigue
    input_df = pd.DataFrame([sample_input])
    pred = model.predict(input_df)[0]

    # Step 2: Optimize task switches (UPDATED FUNCTION)
    optimal_switch, fatigue_pred = optimize_task_switches_practical(sample_input)

    # Step 3: Create range
    lower = max(0, optimal_switch - 1)
    upper = optimal_switch + 1

    print(f"\n--- SAMPLE {i+1} ---")
    print("Current Task Switches:", sample_input["Task_Switches"])
    print("Actual Fatigue:", fatigue_map[actual_label])
    print("Predicted Fatigue:", fatigue_map[pred])
    print("Optimized Task Switch:", optimal_switch)
    print("Recommended Range:", f"{lower}–{upper}")
    print("Fatigue After Optimization:", fatigue_map[fatigue_pred])


--- SAMPLE 1 ---
Current Task Switches: 13
Actual Fatigue: High
Predicted Fatigue: High
Optimized Task Switch: 13
Recommended Range: 12–14
Fatigue After Optimization: High

--- SAMPLE 2 ---
Current Task Switches: 23
Actual Fatigue: High
Predicted Fatigue: High
Optimized Task Switch: 23
Recommended Range: 22–24
Fatigue After Optimization: High

--- SAMPLE 3 ---
Current Task Switches: 2
Actual Fatigue: Low
Predicted Fatigue: Low
Optimized Task Switch: 2
Recommended Range: 1–3
Fatigue After Optimization: Low

--- SAMPLE 4 ---
Current Task Switches: 13
Actual Fatigue: Medium
Predicted Fatigue: Low
Optimized Task Switch: 13
Recommended Range: 12–14
Fatigue After Optimization: Low

--- SAMPLE 5 ---
Current Task Switches: 0
Actual Fatigue: Low
Predicted Fatigue: Low
Optimized Task Switch: 2
Recommended Range: 1–3
Fatigue After Optimization: Low
